# ETL-05 · Extração dos 36 CSVs

- **Etapa:** ETL-05, Extração dos 36 CSVs
- **Responsável:** Pedro Wellington
- **Issue:** ETL-05
- **Entrada:** `data/pessoas_*.csv`
- **Saída:** `data/intermediarios/01_bruto.parquet`

Lê os 36 CSVs mensais (2021 a 2023), junta numa tabela só e salva em parquet. Nenhum dado é alterado.

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS  = RAIZ / "data"                         # CSVs brutos (estão no Git)
INTERM = DADOS / "intermediarios"              # .parquet (o .gitignore já ignora)
INTERM.mkdir(parents=True, exist_ok=True)

SAIDA = INTERM / "01_bruto.parquet"            # resultado desta etapa

## 2. Carregar a entrada

In [2]:
arquivos = sorted(DADOS.glob("pessoas_*.csv"))

lista = []
for a in arquivos:
    d = pd.read_csv(a, encoding='latin-1', sep=';')
    d['ano_origem'] = int(a.stem[-4:])
    d['arquivo_origem'] = a.name
    lista.append(d)

df = pd.concat(lista, ignore_index=True)
print(len(arquivos), 'arquivos'); print('shape:', df.shape); df.head()

36 arquivos
shape: (806977, 33)


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


## 4. Conferências

In [4]:
# colunas originais: as mesmas, na mesma ordem, nos 36 arquivos
base = list(lista[0].columns[:-2])
diferentes = [a.name for a, d in zip(arquivos, lista) if list(d.columns[:-2]) != base]
print(len(base), 'colunas;', len(diferentes), 'arquivos com colunas diferentes')

# total de linhas
print('total de linhas:', len(df))
assert len(df) == 806977

31 colunas; 0 arquivos com colunas diferentes
total de linhas: 806977


## 5.  Salvar a saída

In [5]:
df.to_parquet(SAIDA, index=False)
print('salvo em:', SAIDA, '| shape:', df.shape)

salvo em: c:\Users\pedro\Downloads\Projeto_integração_bd\data\intermediarios\01_bruto.parquet | shape: (806977, 33)


## 6. Resumo

**O que foi feito:** leitura dos 36 arquivos mensais (2021 a 2023) e junção em uma única tabela de 806.977 linhas. Nenhum dado foi alterado. O resultado foi salvo em `data/intermediarios/01_bruto.parquet`.

**Por quê:** as etapas seguintes leem essa tabela, em vez de abrir os 36 CSVs de novo. O parquet ocupa menos espaço e carrega mais rápido.

**O que foi conferido:**
- os 36 arquivos têm as mesmas 31 colunas, na mesma ordem
- o total é de 806.977 linhas

**O que foi acrescentado:** `ano_origem` e `arquivo_origem`, para saber de qual arquivo cada linha veio. A tabela final tem 33 colunas.

**Observação:** o `.parquet` não vai para o Git. Quem precisar dele roda este notebook.